## 🧠 Memory in Agentic Conversations

Memory is essential for any `multi-turn`, `multi-agent`, or long-running conversation. It's one of the core components of AI agents, alongside:
1. Tools
2. Memory
3. Planners

### 🔄 Short-term vs Long-term Memory
AI agents typically use two types of memory:

- **Short-term memory:** Stores recent `turn-by-turn` messages (e.g., the current conversation context).
- **Long-term memory: Stores** high-level summaries or persistent knowledge over time.

📖 For more details, refer to the LangGraph Memory Guide
<p align="center"> <img src="https://blog.langchain.com/content/images/2024/10/short-vs-long.png"width="45%" alt="Short vs Long Memory"/> </p>

### 💾 External Memory Persistence
To support session continuity, especially across multiple threads or users, external storage like Redis can be used to persist chat history.

#### 🔧 Memory Management Patterns

There are two primary patterns for managing memory in multi-agent setups:

- **Agent-level Memory:**
Each agent maintains its own session memory (via separate namespaces or memory savers).
- **Graph-level Memory:**
A shared memory is maintained at the graph level, accessible by all agents.

This is achieved using namespaces or thread IDs in the memory store to separate and manage each session or agent-specific memory.

### Import the model

In [1]:
import LLM_builder

llm = LLM_builder.loadGoogleGenerativeAI()

### Define agent state
But this time this doesn't contain `messages: Annotated[List[HumanMessage | AIMessage]`  as we are using memory at place of this 

In [2]:
from typing import TypedDict

class PlannerState(TypedDict):
    itinerary: str
    city: str
    user_message: str

In [3]:
from langchain.prompts.chat import ChatPromptTemplate, MessagesPlaceholder

itinerary_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful travel assistant. Create a day trip itinerary for {city} based on the user's interests. 
    Follow these instructions:
    1. Use the below chat conversation and the latest input from Human to get the user interests.
    2. Always account for travel time and meal times - if its not possible to do everything, then say so.
    3. If the user hasn't stated a time of year or season, assume summer season in {city} and state this assumption in your response.
    4. If the user hasn't stated a travel budget, assume a reasonable dollar amount and state this assumption in your response.
    5. Provide a brief, bulleted itinerary in chronological order with specific hours of day."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{user_message}"),
])

## Section 1: Custom memory for Graph

### Define custom memory class

Below is an implementation of a custom wrapper over LangGraph’s memory store, where you’re using `InMemoryStore` under the hood but wrapping it with your own interface `CustomMemoryStore`.
- `BaseStore`: Abstract base class for any LangGraph-compatible storage backend.

- `Item`: Represents a stored item (contains key, value, and metadata).

- `InMemoryStore`: Default LangGraph in-memory store implementation.

**You are subclassing BaseStore, meaning your class must implement certain methods (get, put, batch, etc.) that LangGraph expects.** 

##### `get` and `put`
- `get`: Fetches an item from the store.
- `put`: Saves an item into the store.
- `namespace`: Tuple to group keys, helps in avoiding collisions.
- `key`: Specific identifier within the namespace.
- `value`: Dictionary of values you want to store.

##### `batch` and `abatch`
- These methods handle multiple operations at once.
- Each op in ops could be a put or get.

#### FAQs:
`Optional[Item]]` -> "This function might return an object of type `Item`, or it might return `None`."

In [4]:
from langgraph.store.base import BaseStore, Item, Op, Result
from langgraph.store.memory import InMemoryStore
from typing import Any, Iterable, Optional

class CustomMemoryStore(BaseStore):
    def __init__ (self, ext_store):
        self.store = ext_store      # self.store now holds an instance of InMemoryStore.

    # get/aget
    def get(self, namespace: tuple[str, ...], key: str) -> Optional[Item]:
        return self.store.get(namespace, key)

    async def aget(self, namespace: tuple[str, ...], key: str) -> Optional[Item]:
        return await self.store.aget(namespace, key)

    # put/aput
    def put(self, namespace: tuple[str, ...], key: str, value: dict[str, Any]):
        return self.store.put(namespace, key, value)

    async def aput(self, namespace: tuple[str, ...], key: str, value: dict[str, Any]):
        return await self.store.aput(namespace, key, value)

    # bacth/abtach
    def batch(self, ops: Iterable[Op]) -> list[Result]:
        return self.store.batch(ops)

    async def abatch(self, ops: Iterable[Op]) -> list[Result]:
        return await self.store.abatch(ops)


In [5]:
in_memory_store = CustomMemoryStore(InMemoryStore())
namespace_u = ("chat_messages", "user_id_1")
key_u = "user_id_1"

in_memory_store.put(namespace_u, key_u, {"user_id": "user_id_1", "message": "Hello LangGraph!"})

item_u = in_memory_store.get(namespace_u, key_u)
print(item_u.value)
# print(item_u.value, item_u.value['data'])

in_memory_store.list_namespaces()

{'user_id': 'user_id_1', 'message': 'Hello LangGraph!'}


[('chat_messages', 'user_id_1')]

## Create similar graph as before

In [ ]:
from langchain_core.runnables.config import RunnableConfig
from langchain_core.messages import HumanMessage, AIMessage

def input_interests(state: PlannerState, config: RunnableConfig, *, store: BaseStore) -> PlannerState:
    return {
        **state,
    }

def create_itinerary(state: PlannerState, config: RunnableConfig, *, store: BaseStore) -> PlannerState:
    # get the history from the store
    user_u = f"user_id_{config['configurable']['thread_id']}"
    namespace_u = ("chat_messages", user_u)
    store_item = store.get(namespace=namespace_u, key=user_u)
    chat_history_messages = store_item.value['data'] if store_item else []
    print(user_u,chat_history_messages)

    response = llm.invoke(itinerary_prompt.format_messages(city=state['city'], user_message=state['user_message'], chat_history=chat_history_messages))
    print("\nFinal Itinerary:")
    print(response.content)

    # add back to the store
    store.put(namespace=namespace_u, key=user_u, value={"data":chat_history_messages+[HumanMessage(content=state['user_message']),AIMessage(content=response.content)]})
    
    return {
        **state,
        "itinerary": response.content
    }

In [7]:
from langgraph.graph import StateGraph, END

in_memory_store_n = CustomMemoryStore(InMemoryStore())

workflow = StateGraph(PlannerState)

#workflow.add_node("input_city", input_city)
workflow.add_node("input_interests", input_interests)
workflow.add_node("create_itinerary", create_itinerary)

workflow.set_entry_point("input_interests")

#workflow.add_edge("input_city", "input_interests")
workflow.add_edge("input_interests", "create_itinerary")
workflow.add_edge("create_itinerary", END)


app = workflow.compile(store=in_memory_store_n)

In [8]:
def run_travel_planner(user_request: str, config_dict: dict):
    print(f"Current User Request: {user_request}\n")
    init_input = {"user_message": user_request,"city" : "Seattle"}

    for output in app.stream(init_input, config=config_dict, stream_mode="values"):
        pass  # The nodes themselves now handle all printing

config = {"configurable": {"thread_id": "1"}}

user_request = "Can you create a itinerary for a day trip in california with boating and swimming options.  I need a complete plan that budgets for travel time and meal time."
run_travel_planner(user_request, config)


Current User Request: Can you create a itinerary for a day trip in california with boating and swimming options.  I need a complete plan that budgets for travel time and meal time.

user_id_1 []

Final Itinerary:
I can do that! However, since you're asking for a day trip itinerary in California, it would be helpful to know where you're starting from to factor in travel time. Also, let me know what your interests are in California - the state is so diverse. Do you want to be in Southern California, or Northern California? Do you want to see a specific city or attraction? And what is your budget?


In [9]:
config = {"configurable": {"thread_id": "1"}}

user_request = "Can you add itinerary for white water rafting to this"
run_travel_planner(user_request, config)

Current User Request: Can you add itinerary for white water rafting to this

user_id_1 [HumanMessage(content='Can you create a itinerary for a day trip in california with boating and swimming options.  I need a complete plan that budgets for travel time and meal time.', additional_kwargs={}, response_metadata={}), AIMessage(content="I can do that! However, since you're asking for a day trip itinerary in California, it would be helpful to know where you're starting from to factor in travel time. Also, let me know what your interests are in California - the state is so diverse. Do you want to be in Southern California, or Northern California? Do you want to see a specific city or attraction? And what is your budget?", additional_kwargs={}, response_metadata={})]

Final Itinerary:
Okay, I can create a day trip itinerary that includes boating, swimming, and white water rafting. However, I still need a starting location, so I'm going to switch gears and create an itinerary for a day trip in

In [10]:
print(in_memory_store_n.list_namespaces())
print(in_memory_store_n.get(('chat_messages', 'user_id_1'),'user_id_1').value)

[('chat_messages', 'user_id_1')]
{'data': [HumanMessage(content='Can you create a itinerary for a day trip in california with boating and swimming options.  I need a complete plan that budgets for travel time and meal time.', additional_kwargs={}, response_metadata={}), AIMessage(content="I can do that! However, since you're asking for a day trip itinerary in California, it would be helpful to know where you're starting from to factor in travel time. Also, let me know what your interests are in California - the state is so diverse. Do you want to be in Southern California, or Northern California? Do you want to see a specific city or attraction? And what is your budget?", additional_kwargs={}, response_metadata={}), HumanMessage(content='Can you add itinerary for white water rafting to this', additional_kwargs={}, response_metadata={}), AIMessage(content="Okay, I can create a day trip itinerary that includes boating, swimming, and white water rafting. However, I still need a starting l

## Section 2: Each agent have it's own memory

- `from langchain_core.chat_history import InMemoryChatMessageHistory`
    - Imports an in-memory store to hold chat messages — useful for prototyping or short-lived sessions.

    - It keeps a list of past HumanMessage and AIMessage objects in RAM (not persisted to file or DB).

- `from langchain_core.runnables.history import RunnableWithMessageHistory`
    - Imports a wrapper class that adds memory capability to any Runnable (like an LLM chain). It lets your chain maintain and use chat history across multiple calls.

- `chain = itinerary_prompt | llm`
    - Using the | operator means: Send the output of `itinerary_prompt` as input to `llm`

-  `def get_history(): return history`
    - A simple getter function that returns the same in-memory history.

    - LangChain needs this function to dynamically fetch the history when needed.

In [11]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

chain = itinerary_prompt | llm 

history = InMemoryChatMessageHistory()
def get_history():
    return history

wrapped_chain = RunnableWithMessageHistory(
    chain,
    get_history,
    history_messages_key="chat_history",
)

In [12]:
def create_itinerary(state: PlannerState, config: RunnableConfig, *, store: BaseStore) -> PlannerState:
    #- each agent manages it's memory
    response = wrapped_chain.invoke({"city": state['city'], "user_message": state['user_message'], "input": state['user_message']} )
    print("\nFinal Itinerary:")
    print(response.content)
    
    return {
        **state,
        "itinerary": response.content
    }

In [13]:
workflow = StateGraph(PlannerState)

#workflow.add_node("input_city", input_city)
workflow.add_node("input_interests", input_interests)
workflow.add_node("create_itinerary", create_itinerary)

workflow.set_entry_point("input_interests")

#workflow.add_edge("input_city", "input_interests")
workflow.add_edge("input_interests", "create_itinerary")
workflow.add_edge("create_itinerary", END)


app = workflow.compile()

In [14]:
def run_travel_planner(user_request: str, config_dict: dict):
    print(f"Current User Request: {user_request}\n")
    init_input = {"user_message": user_request,"city" : "Seattle"}

    for output in app.stream(init_input, config=config_dict, stream_mode="values"):
        pass  # The nodes themselves now handle all printing

config = {"configurable": {"thread_id": "1"}}

user_request = "Can you create a itinerary for boating, swim. Need a complete plan"
run_travel_planner(user_request, config)

Current User Request: Can you create a itinerary for boating, swim. Need a complete plan


Final Itinerary:
Okay, I can help you create a day trip itinerary in Seattle focused on boating and swimming. Since you haven't specified a time of year, I'll assume this is for the summer season in Seattle. I'll also assume a reasonable budget of $200 for the day, which should cover rentals and food.

Here's a possible itinerary:

*   **9:00 AM:** Arrive at **Lake Union**. This will give you time to find parking and get ready for the day.

*   **9:30 AM - 12:30 PM:** **Boat Rental and Lake Exploration.** Rent a boat from one of the rental places on Lake Union. You can choose from various options like kayaks, paddleboards, or small motorboats, depending on your preference and budget. Enjoy exploring the lake, taking in views of the Seattle skyline, Gas Works Park, and houseboats.

*   **12:30 PM - 1:30 PM:** **Lunch at Ivar's Acres of Clams** Enjoy some seafood with a view.

*   **2:00 PM - 5:00 

In [15]:
user_request = "Can you add white water rafting to this itinerary"
run_travel_planner(user_request, config)

Current User Request: Can you add white water rafting to this itinerary


Final Itinerary:
Okay, I can add white water rafting to your Seattle day trip itinerary. However, since whitewater rafting locations are outside of Seattle and require significant travel time, incorporating it into the same day with boating and swimming will be challenging. Here's a revised itinerary that prioritizes whitewater rafting and includes one of your other interests:

Given the travel time, we'll have to drop the lake boating activity to make this work.

*   **7:00 AM:** Depart from Seattle. The drive to a suitable rafting location (like the Skykomish River) will take approximately 1.5 to 2 hours. Pack snacks and drinks for the drive.

*   **9:00 AM - 12:30 PM:** **White Water Rafting on the Skykomish River.** Arrive at the rafting outfitter, get geared up, and enjoy a thrilling whitewater rafting experience. Most trips last around 3 hours, including instruction and the actual rafting time.

*   **12:30